In [ ]:
# VGG_RESEARCH_ONE_CELL_V1: Colab GPU, paste and run this whole cell.
DATA_DIR = '/content/drive/MyDrive/T2_data'  # Change only this path.
FINETUNE_BUDGET_MINUTES = 35  # Both candidates combined; excludes preparation/export.
MAX_EPOCHS_PER_CANDIDATE = 12
PATIENCE = 4
BATCH_SIZE = 8
SEED = 42
IMAGE_SIZE = (336, 224)  # Height, width; same as the original VGG experiment.
LABEL_SMOOTHING = .03
NUM_WORKERS = 2
# Audited from the first report's predictions; same exact threshold search as below.
BASELINE_EXACT_BA = .7580882352941176
BASELINE_EXACT_THRESHOLD = .0055766392033547

import importlib.util, subprocess, sys
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'pillow',
          'matplotlib': 'matplotlib', 'tqdm': 'tqdm'}
missing = [p for m, p in needed.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

import gc, hashlib, json, random, shutil, time, zipfile
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
import importlib.metadata
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from google.colab import drive, files

if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 GPU 런타임을 선택하고 다시 실행하세요.')
assert FINETUNE_BUDGET_MINUTES > 0 and MAX_EPOCHS_PER_CANDIDATE >= 1
drive.mount('/content/drive')
# Run in a fresh runtime if a previous model is still occupying GPU memory.
for name in ['model', 'optimizer', 'scaler', 'checkpoint', 'train_loader', 'valid_loader']:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
device = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))

source = Path(DATA_DIR)
if not source.is_dir():
    raise FileNotFoundError(f'DATA_DIR을 실제 Drive 데이터 폴더로 바꾸세요: {source}')
csvs = [source / 'labels.csv'] if (source / 'labels.csv').is_file() else list(source.rglob('labels.csv'))
if len(csvs) != 1:
    raise ValueError('DATA_DIR 아래 labels.csv가 정확히 하나 있어야 합니다.')
label_file = csvs[0]
frame = pd.read_csv(label_file, dtype={'sample_id': str, 'patient_group': str})
columns = ['sample_id', 'path', 'label', 'patient_group']
assert set(columns).issubset(frame.columns)
assert frame.sample_id.is_unique and not frame[columns].isna().any().any()
assert set(frame.label.unique()) == {0, 1}
assert frame.patient_group.nunique() == 27 and len(frame) == 2948
for p in frame.path:
    resolved = (label_file.parent / str(p)).resolve()
    if not resolved.is_relative_to(label_file.parent.resolve()) or not resolved.is_file():
        raise ValueError(f'이미지 경로를 확인하세요: {p}')

VALID_GROUPS = ['G06', 'G07', 'G09', 'G14', 'G17', 'G20']
frame['split'] = np.where(frame.patient_group.isin(VALID_GROUPS), 'valid', 'train')
train_df = frame[frame.split.eq('train')].reset_index(drop=True)
valid_df = frame[frame.split.eq('valid')].reset_index(drop=True)
assert len(train_df) == 2349 and len(valid_df) == 599
assert not set(train_df.patient_group) & set(valid_df.patient_group)
assert set(train_df.label) == set(valid_df.label) == {0, 1}
label_hash = hashlib.sha256(frame[columns].to_csv(index=False).encode()).hexdigest()
stamp = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
out = source.parent / 'task2_vgg_research' / stamp
out.mkdir(parents=True)
local = Path('/content') / f'vgg_research_{stamp}'
local.mkdir()
frame.to_csv(out / 'patient_split.csv', index=False)
print('Train: 21 patients / 2349 images; Validation: 6 patients / 599 images')
print('Validation patients:', VALID_GROUPS)
print('Output:', out)

# Decode and resize once. Original files stay on Drive. About 0.67 GB for RGB cache.
def read_resized(relative_path):
    with Image.open(label_file.parent / str(relative_path)) as im:
        return im.convert('RGB').resize((IMAGE_SIZE[1], IMAGE_SIZE[0]), Image.Resampling.BILINEAR)

print('이미지를 한 번 읽어 메모리에 캐시합니다. 이후 epoch마다 Drive에서 읽지 않습니다.')
with ThreadPoolExecutor(max_workers=4) as pool:
    cached = list(tqdm(pool.map(read_resized, frame.path.tolist()), total=len(frame), desc='Cache images'))
cache = dict(zip(frame.sample_id, cached))
del cached
MEAN = [.485, .456, .406]
STD = [.229, .224, .225]
valid_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(.5), transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=.1, contrast=.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class CachedImages(Dataset):
    def __init__(self, rows, training=False):
        self.rows = rows.reset_index(drop=True)
        self.transform = train_transform if training else valid_transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        return self.transform(cache[row.sample_id]), int(row.label), index

def loader(rows, training=False, batch_size=BATCH_SIZE, seed=SEED):
    return DataLoader(CachedImages(rows, training), batch_size=batch_size,
                      shuffle=training, num_workers=NUM_WORKERS, pin_memory=True,
                      generator=torch.Generator().manual_seed(seed),
                      persistent_workers=False)

valid_loader = loader(valid_df)

def sample_weights(rows, patient_balance=False):
    y = rows.label.to_numpy(int)
    result = np.zeros(len(rows), dtype=np.float64)
    counts = rows.groupby(['patient_group', 'label']).size().to_dict()
    for c in [0, 1]:
        mask = y == c
        adjustment = np.ones(int(mask.sum()), dtype=np.float64)
        if patient_balance:
            bucket_sizes = np.array([counts[(g, c)] for g in rows.loc[mask, 'patient_group']], dtype=float)
            adjustment = 1. / np.sqrt(bucket_sizes)
            adjustment /= adjustment.mean()
        result[mask] = len(rows) / (2 * mask.sum()) * adjustment
    return result.astype(np.float32)

def metric_pack(y, p):
    """All ROC cuts, tied scores kept together; threshold uses >= in all paths."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=np.float64)
    assert len(y) == len(p) and set(np.unique(y)) == {0, 1}
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    n1, n0 = int(y.sum()), int((y == 0).sum())
    order = np.argsort(-p, kind='stable')
    ys, ps = y[order], p[order]
    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]), len(ps) - 1]
    tp = np.r_[0, np.cumsum(ys)[ends]]
    fp = np.r_[0, np.cumsum(1 - ys)[ends]]
    tpr, fpr = tp / n1, fp / n0
    auc = float(np.trapezoid(tpr, fpr))
    ba = .5 * (tpr + 1 - fpr)
    thresholds = np.r_[np.nextafter(ps[0], np.inf),
                       (ps[ends[:-1]] + ps[ends[:-1] + 1]) / 2., ps[-1]]
    best = np.flatnonzero(np.isclose(ba, ba.max(), rtol=0, atol=1e-12))
    j = int(best[np.argmin(np.abs(thresholds[best] - .5))])
    threshold = float(thresholds[j])
    pred = p >= threshold
    pred05 = p >= .5
    recall0 = float((~pred[y == 0]).mean())
    recall1 = float(pred[y == 1].mean())
    return dict(valid_ba_tuned=.5 * (recall0 + recall1), valid_auroc=auc,
                threshold=threshold, recall0_tuned=recall0, recall1_tuned=recall1,
                valid_ba_05=float(.5 * ((~pred05[y == 0]).mean() + pred05[y == 1].mean())))

def selection_key(metrics):
    return (round(float(metrics['valid_ba_tuned']), 10),
            round(float(metrics['valid_auroc']), 10))

config = dict(architecture='torchvision.vgg11', initialization='IMAGENET1K_V1',
              seed=SEED, image_size=list(IMAGE_SIZE), normalize_mean=MEAN, normalize_std=STD,
              class_names=['absent', 'present'], positive_class=1, valid_groups=VALID_GROUPS,
              train_groups=sorted(train_df.patient_group.unique().tolist()), label_hash=label_hash,
              batch_size=BATCH_SIZE, label_smoothing=LABEL_SMOOTHING,
              brightness=.1, contrast=.1, horizontal_flip=.5, rotation_degrees=5,
              fine_tune_budget_minutes=FINETUNE_BUDGET_MINUTES,
              max_epochs_per_candidate=MAX_EPOCHS_PER_CANDIDATE, patience=PATIENCE,
              head_epochs=20, head_selection='AUROC then tuned BA',
              checkpoint_selection='tuned Validation BA then AUROC; not independent Test',
              inference='one VGG, one original resized frame, no ensemble/TTA',
              resume='best weights saved periodically; no automatic optimizer-state resume')
(out / 'config.json').write_text(json.dumps(config, indent=2))
references = {
    'LP-FT': 'https://arxiv.org/abs/2202.10054',
    'LP-FT implementation': 'https://github.com/AnanyaKumar/transfer_learning',
    'Medical fine-tuning comparison': 'https://arxiv.org/abs/2406.10050',
    'Label smoothing': 'https://arxiv.org/abs/1906.02629',
    'Channels-last AMP': 'https://docs.pytorch.org/tutorials/intermediate/memory_format_tutorial.html',
    'Patient weighting': 'Project-specific hypothesis; no performance guarantee',
}
(out / 'references.json').write_text(json.dumps(references, indent=2))

seed_all(SEED)
model = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(4096, 2)
model = model.to(device=device, memory_format=torch.channels_last)
for parameter in model.parameters():
    parameter.requires_grad = False

# Use this VGG itself to compute fixed features once; no other neural network.
@torch.inference_mode()
def extract_features(rows):
    model.eval()  # Frozen classifier dropout is OFF for a clean linear probe.
    chunks, targets = [], []
    for images, labels, _ in tqdm(loader(rows, batch_size=16), desc='VGG feature cache'):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16):
            x = model.features(images)
            x = model.avgpool(x).flatten(1)
            for layer in list(model.classifier.children())[:-1]:
                x = layer(x)
        chunks.append(x.float().cpu())
        targets.append(labels)
    return torch.cat(chunks), torch.cat(targets)

print('출력층 준비 학습: VGG 특징을 한 번만 추출합니다.')
train_features, head_y_cpu = extract_features(train_df)
valid_features, valid_y_cpu = extract_features(valid_df)
Xtr, Xva = train_features.to(device), valid_features.to(device)
ytr, yva = head_y_cpu.to(device), valid_y_cpu.to(device)
del train_features, valid_features
head = model.classifier[-1]
for parameter in head.parameters():
    parameter.requires_grad = True
head_w = torch.from_numpy(sample_weights(train_df)).to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
best_head_key = (-1., -1.)
best_head_state = None
head_history = []
for epoch in range(1, 21):
    indices = torch.randperm(len(ytr), device=device)
    for batch in indices.split(128):
        optimizer.zero_grad(set_to_none=True)
        losses = F.cross_entropy(head(Xtr[batch]), ytr[batch], reduction='none',
                                 label_smoothing=LABEL_SMOOTHING)
        loss = (losses * head_w[batch]).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(head.parameters(), 5.)
        optimizer.step()
    with torch.no_grad():
        p = head(Xva).softmax(1)[:, 1].cpu().numpy().astype(float)
    m = metric_pack(valid_y_cpu.numpy(), p)
    head_history.append(dict(epoch=epoch, **m))
    key = (m['valid_auroc'], m['valid_ba_tuned'])
    if key > best_head_key:
        best_head_key = key
        best_head_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    if epoch == 1 or epoch % 5 == 0:
        print(f"Head {epoch:02d}: BA={m['valid_ba_tuned']:.4f}, AUC={m['valid_auroc']:.4f}")
head.load_state_dict(best_head_state)
pd.DataFrame(head_history).to_csv(out / 'head_history.csv', index=False)
del Xtr, Xva, ytr, yva, head_w, optimizer, head, best_head_state
gc.collect()
torch.cuda.empty_cache()
initial_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

@torch.inference_mode()
def evaluate_current():
    model.eval()
    probabilities, targets = [], []
    loss_sum = 0.
    for images, y, _ in valid_loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(images)
        # Unweighted, unsmoothed validation CE for reporting, not training loss.
        loss_sum += float(F.cross_entropy(logits.float(), y, reduction='sum').item())
        probabilities.append(logits.float().softmax(1)[:, 1].cpu().numpy())
        targets.append(y.cpu().numpy())
    y = np.concatenate(targets)
    p = np.concatenate(probabilities).astype(float)
    return dict(**metric_pack(y, p), valid_ce=loss_sum / len(y)), y, p

def checkpoint_payload(candidate, info):
    return dict(model_state={k: v.detach().cpu() for k, v in model.state_dict().items()},
                config=dict(config, candidate=candidate), best_metrics=info,
                threshold=float(info['threshold']))

def atomic_copy(src, dst):
    temporary = dst.with_suffix(dst.suffix + '.tmp')
    shutil.copy2(src, temporary)
    temporary.replace(dst)

def report_predictions(folder, info, y, p):
    predictions = valid_df[['sample_id', 'patient_group', 'label']].copy()
    assert np.array_equal(predictions.label.to_numpy(int), y)
    predictions['probability'] = p
    predictions['prediction_05'] = (p >= .5).astype(int)
    predictions['prediction_tuned'] = (p >= info['threshold']).astype(int)
    predictions.to_csv(folder / 'valid_predictions.csv', index=False)
    patient_rows = []
    for g, part in predictions.groupby('patient_group'):
        yy, pp = part.label.to_numpy(int), part.prediction_tuned.to_numpy(int)
        r0, r1 = float((pp[yy == 0] == 0).mean()), float((pp[yy == 1] == 1).mean())
        patient_rows.append(dict(patient_group=g, images=len(part), positives=int(yy.sum()),
                                 ba_tuned=(r0 + r1) / 2, recall0=r0, recall1=r1))
    pd.DataFrame(patient_rows).to_csv(folder / 'patient_metrics.csv', index=False)
    return float(np.mean([r['ba_tuned'] for r in patient_rows]))

candidates = [('lpft_class', False), ('lpft_patient', True)]
comparison = []
history_all = []
for candidate, patient_balance in candidates:
    # Independent candidates share identical starting weights; no prediction averaging.
    seed_all(SEED)
    model.load_state_dict(initial_state)
    for parameter in model.parameters():
        parameter.requires_grad = True
    candidate_dir = out / candidate
    candidate_dir.mkdir()
    best_file = local / f'{candidate}.pt'
    train_loader = loader(train_df, training=True, seed=SEED)
    weights_np = sample_weights(train_df, patient_balance)
    weights = torch.from_numpy(weights_np).to(device)
    weight_frame = train_df[['sample_id', 'patient_group', 'label']].copy()
    weight_frame['loss_weight'] = weights_np
    weight_frame.to_csv(candidate_dir / 'train_loss_weights.csv', index=False)

    # Keep the entire original VGG classifier. Smaller LR protects early features.
    groups = []
    regions = [('early', model.features[:11], 2e-6),
               ('late', model.features[11:], 1e-5),
               ('fc', model.classifier[:-1], 5e-6),
               ('output', model.classifier[-1], 5e-5)]
    for region, module, lr in regions:
        decay, no_decay = [], []
        for name, parameter in module.named_parameters():
            (no_decay if parameter.ndim == 1 or name.endswith('bias') else decay).append(parameter)
        for params, wd in [(decay, 1e-4), (no_decay, 0.)]:
            if params:
                groups.append(dict(params=params, lr=lr, weight_decay=wd, region=region))
    optimizer = torch.optim.AdamW(groups)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=.5, patience=2, min_lr=1e-7)
    metrics, _, _ = evaluate_current()
    best_info = dict(candidate=candidate, epoch=0, phase='linear_probe', **metrics)
    best_key = selection_key(metrics)
    torch.save(checkpoint_payload(candidate, best_info), best_file)
    atomic_copy(best_file, candidate_dir / 'vgg.pt')
    bad_epochs = 0
    candidate_rows = []
    # Each candidate has an equal time allowance; export time may add to wall time.
    deadline = time.perf_counter() + FINETUNE_BUDGET_MINUTES * 60 / len(candidates)
    print(f'\nCandidate: {candidate}; patient balance={patient_balance}')
    for epoch in range(1, MAX_EPOCHS_PER_CANDIDATE + 1):
        if time.perf_counter() >= deadline:
            print('후보별 시간 예산에 도달했습니다.')
            break
        started = time.perf_counter()
        # Short transition, then train all blocks (not the previous last-block-only setting).
        for parameter in model.features[:11].parameters():
            parameter.requires_grad = epoch > 2
        model.train()
        numerator, denominator, batches = 0., 0., 0
        budget_hit = False
        for images, targets, indices in tqdm(train_loader, desc=f'{candidate} {epoch}', leave=False):
            if time.perf_counter() >= deadline:
                budget_hit = True
                break
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            targets = targets.to(device, non_blocking=True)
            w = weights[indices.to(device, non_blocking=True)]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(images)
                losses = F.cross_entropy(logits, targets, reduction='none',
                                         label_smoothing=LABEL_SMOOTHING)
                loss = (losses * w).mean()
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError('학습 손실이 유한하지 않습니다. 실행을 중단합니다.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.)
            scaler.step(optimizer)
            scaler.update()
            numerator += float((losses.detach() * w).sum().item())
            denominator += float(w.sum().item())
            batches += 1
        if batches == 0:
            break
        metrics, _, _ = evaluate_current()
        row = dict(candidate=candidate, epoch=epoch,
                   phase='late_blocks_and_classifier' if epoch <= 2 else 'all_layers',
                   partial_epoch=budget_hit, batches=batches,
                   train_weighted_smoothed_ce=numerator / denominator,
                   **metrics, seconds=round(time.perf_counter() - started, 2))
        candidate_rows.append(row)
        history_all.append(row)
        pd.DataFrame(candidate_rows).to_csv(candidate_dir / 'history.csv', index=False)
        if selection_key(metrics) > best_key:
            best_key, best_info = selection_key(metrics), dict(row)
            torch.save(checkpoint_payload(candidate, best_info), best_file)
            bad_epochs = 0
        else:
            bad_epochs += 1
        scheduler.step(metrics['valid_ba_tuned'])
        if epoch % 3 == 0:
            atomic_copy(best_file, candidate_dir / 'vgg.pt')
        print(f"{candidate} {epoch:02d}: BA@.5={metrics['valid_ba_05']:.4f}, "
              f"BA tuned={metrics['valid_ba_tuned']:.4f}, AUC={metrics['valid_auroc']:.4f}, "
              f"best BA={best_key[0]:.4f}, {row['seconds']:.0f}s", flush=True)
        # Give full-network adaptation at least two epochs before stopping.
        if budget_hit or (epoch >= 4 and bad_epochs >= PATIENCE):
            print('시간 예산 또는 조기 종료 조건에 도달했습니다.')
            break

    atomic_copy(best_file, candidate_dir / 'vgg.pt')
    checkpoint = torch.load(best_file, map_location='cpu', weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    del checkpoint
    final_metrics, y, p = evaluate_current()
    assert np.isclose(final_metrics['valid_ba_tuned'], best_info['valid_ba_tuned'], atol=1e-10)
    final_info = dict(best_info, **{'mean_patient_ba': report_predictions(candidate_dir, best_info, y, p)})
    final_info.update(train_images=len(train_df), valid_images=len(valid_df),
                      weights_path=str(candidate_dir / 'vgg.pt'),
                      note='Same validation used for epoch, threshold and candidate selection; not independent Test.')
    (candidate_dir / 'metrics.json').write_text(json.dumps(final_info, indent=2))
    comparison.append(final_info)
    pd.DataFrame(comparison).to_csv(out / 'comparison.csv', index=False)
    del optimizer, scaler, scheduler, train_loader, weights
    gc.collect()
    torch.cuda.empty_cache()

selected = max(comparison, key=selection_key)
selection = dict(selected_candidate=selected['candidate'], weights_path=selected['weights_path'],
                 threshold=selected['threshold'], valid_ba_tuned=selected['valid_ba_tuned'],
                 valid_auroc=selected['valid_auroc'],
                 historical_baseline_ba=.7557247899159664,
                 baseline_ba_with_same_exact_threshold_search=BASELINE_EXACT_BA,
                 baseline_exact_threshold=BASELINE_EXACT_THRESHOLD,
                 historical_baseline_auroc=.8269957983193277,
                 exceeds_historical_ba=bool(selected['valid_ba_tuned'] > BASELINE_EXACT_BA),
                 note='Best NEW candidate only. Original baseline weights are preserved. No ensemble or full-data refit.')
(out / 'selected_candidate.json').write_text(json.dumps(selection, indent=2))
(out / 'requirements_actual.txt').write_text('\n'.join(
    f'{p}=={importlib.metadata.version(p)}' for p in ['torch', 'torchvision', 'numpy', 'pandas', 'pillow', 'matplotlib', 'tqdm']) + '\n')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for candidate in [c[0] for c in candidates]:
    h = pd.DataFrame([r for r in history_all if r['candidate'] == candidate])
    if len(h):
        axes[0].plot(h.epoch, h.valid_ba_tuned, marker='.', label=candidate)
        axes[1].plot(h.epoch, h.valid_auroc, marker='.', label=candidate)
axes[0].axhline(BASELINE_EXACT_BA, color='gray', ls='--', label='baseline, same exact threshold search')
axes[1].axhline(.8269957983193277, color='gray', ls='--', label='historical baseline AUC')
for ax, ylabel in zip(axes, ['Validation tuned BA', 'Validation AUROC']):
    ax.set(xlabel='Fine-tuning epoch', ylabel=ylabel)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out / 'learning_curves.png', dpi=150)
plt.show()
plt.close(fig)

# Save a separate, minimal inference script for the selected single VGG model.
inference_source = r'''import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torchvision import models, transforms

def predict_csv(weights_path, csv_path, image_root, output_csv='predictions.csv'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(weights_path, map_location='cpu', weights_only=True)
    config = checkpoint['config']
    model = models.vgg11(weights=None)
    model.classifier[-1] = torch.nn.Linear(4096, 2)
    model.load_state_dict(checkpoint['model_state'])
    model = model.to(device=device, memory_format=torch.channels_last).eval()
    threshold = float(checkpoint['threshold'])
    frame = pd.read_csv(csv_path, dtype={'sample_id': str})
    transform = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize(config['normalize_mean'], config['normalize_std'])])
    height, width = config['image_size']
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(frame), 8):
            images = []
            for p in frame.iloc[start:start+8].path:
                with Image.open(Path(image_root) / str(p)) as im:
                    im = im.convert('RGB').resize((width, height), Image.Resampling.BILINEAR)
                    images.append(transform(im))
            batch = torch.stack(images).to(device=device, memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(batch)
            probabilities.extend(logits.float().softmax(1)[:, 1].cpu().numpy().tolist())
    result = frame[['sample_id']].copy()
    result['probability'] = probabilities
    result['prediction'] = (np.asarray(probabilities) >= threshold).astype(int)
    result.to_csv(output_csv, index=False)
    return result
'''
(out / 'inference.py').write_text(inference_source)
source_text = globals().get('_VGG_EXECUTED_SOURCE')
if source_text is None:
    try:
        source_text = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        source_text = '# Training source is in the original Colab cell.\n'
(out / 'executed_cell.py').write_text(source_text)
small_zip = out / 'vgg_research_report.zip'
with zipfile.ZipFile(small_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file() and p.suffix in {'.json', '.csv', '.txt', '.png', '.py'}:
            z.write(p, str(p.relative_to(out)))
print('\n', pd.DataFrame(comparison)[['candidate', 'epoch', 'valid_ba_05', 'valid_ba_tuned', 'valid_auroc']].to_string(index=False))
print('\nBest NEW candidate:', json.dumps(selection, indent=2))
if not selection['exceeds_historical_ba']:
    print('같은 임계값 탐색으로 계산한 첫 VGG의 BA 0.7581을 넘지 못했습니다. 첫 모델을 유지하세요.')
print('가중치는 Drive에 저장했습니다. 보고서 ZIP에는 가중치를 넣지 않았습니다.')
print('자동 재개 기능은 없습니다. 최고 가중치는 3 epoch마다 및 후보 종료 시 Drive에 저장됩니다.')
for _gpu_name in ['initial_state', 'model', 'cache', 'regions', 'groups', 'module',
                  'parameter', 'decay', 'no_decay', 'params', 'logits', 'losses',
                  'loss', 'w', 'images', 'targets', 'batch', 'indices']:
    globals().pop(_gpu_name, None)
gc.collect()
torch.cuda.empty_cache()
files.download(str(small_zip))
